In [1]:
import pandas as pd
import numpy as np
import datetime

In [2]:
twayFreq = 60
localFreq = 30
outputFolderPath = './output/future/frequencyMatrix/T'+str(twayFreq)+'_L'+str(localFreq)+'/'

In [3]:
gtfsFolderPath = './output/future/road/4TOD/'
shapes = pd.read_csv(gtfsFolderPath + 'shapes.txt')
stops = pd.read_csv(gtfsFolderPath + 'stops.txt')
stopsByRoute = pd.read_csv('StopsByRoute_TwaysAndLocalRoutes_4TOD.csv')
schedules = pd.read_csv('routeSchedules_TwaysAndLocalRoutes_4TOD.csv')

In [4]:
stops = stops.drop('Unnamed: 0', axis=1)

In [5]:
stopsByRoute.drop('shape_id', axis=1, inplace=True)
stopsByRoute = stopsByRoute.rename(columns={'parent_route':'shape_id'})
stopsByRoute.drop_duplicates(inplace=True)
stopsByRoute.head()

,begin,end,stop_id,stop_name,stop_lat,stop_lon,distAlongR,shape_id
0,1,428,20200814260,10thAv_AerotropolisSouth_b,-33.934483,150.740965,47.945026,10th-ASTH-CSLA
1,1,428,20200814198,Rossmore-SW-WE,-33.927395,150.769325,3972.953166,10th-ASTH-CSLA
2,1,428,20200814202,Rossmore-SE-WE,-33.929194,150.783205,5271.874157,10th-ASTH-CSLA
3,1,428,20200814210,Austral-SW-WE,-33.931200,150.798671,6720.249897,10th-ASTH-CSLA
4,1,428,20200814214,Austral-SE-WE,-33.933048,150.811829,7954.242685,10th-ASTH-CSLA


In [6]:
schedules.head()

,route,route_short_name,from,to,off_peak_tt,first_start_M_F,first_start_Sat,first_start_Sun,last_start_M_F,last_start_Sat,last_start_Sun,avg_freq_M_F,avg_freq_Sat,avg_freq_Sun,nTrips_M_F,nTrips_Sat,nTrips_Sun,Notes
0,AERO-MG,AERO-MG,Aerotropolis,Middleton Grange,25,4:30,NaN,NaN,23:30,NaN,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN
1,MG-AERO,MG-AERO,Middleton Grange,Aerotropolis,25,4:30,NaN,NaN,23:30,NaN,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN
2,10th-ASTH-CSLA,10th Av Tway,Aerotropolis South,Casula,38,4:30,NaN,NaN,23:30,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN
3,10th-CSLA-ASTH,10th Av Tway,Casula,Aerotropolis South,38,4:30,NaN,NaN,23:30,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN
4,15th-AERO-LVPL,15th Av Tway,Aerotropolis,Liverpool,40,4:30,NaN,NaN,23:30,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
schedules['routeType'] = schedules.apply(lambda x: 'MG' if 'MG' in x['route'] else 'Local' if any(s in x['route'] for s in ['Austral','Rossmore','KempsCreek']) else 'T-Way', axis=1)
schedules['routeType'].value_counts()

Local    48
T-Way    16
MG        2
Name: routeType, dtype: int64

In [8]:
schedules['avg_freq_M_F'] = schedules.apply(lambda x: twayFreq if x['routeType']=='T-Way' else localFreq if x['routeType']=='Local' else 10, axis=1)
schedules['avg_freq_M_F'].value_counts()

30    48
60    16
10     2
Name: avg_freq_M_F, dtype: int64

Agency ids and names:
* "SLR","Sydney Light Rail"
* "SMNW","Sydney Metro"
* "x0001","Sydney Trains"
* "2435","Transit Systems"

Route desc, type, color, text color, exact times:
* "Sydney Metro Network","1","008C96","FFFFFF","0"
* "Sydney Trains Network","2","F6891F","FFFFFF","1"
* "Sydney Light Rail Network","0","EE343F","FFFFFF","0"
* "Sydney Buses Network","700","00B5EF","FFFFFF","1"

In [9]:
def getRoutesTripsStoptimes(schedules, shapes, stopsByRoute, stops):
    
    schedules = schedules.loc[schedules['avg_freq_M_F']>0]
    
    # routes.txt
    routes = schedules[['route','from','to','route_short_name']]
    routes['agency_id'] = "2435"
    routes = routes.rename(columns={'route':'route_id'})
    routes['route_long_name'] = routes['route_id'].astype(str).str[0] + ' ' + routes['from'] + ' - ' + routes['to']
    routes['route_desc'] = "Sydney Buses Network"
    routes['route_type'] = "700"
    routes['route_color'] = "00B5EF"
    routes['route_text_color'] = "FFFFFF"
    routes['exact_times'] = "1"
    routes = routes.drop(['from','to'], axis=1)
    routes = routes[['route_id','agency_id','route_short_name','route_long_name','route_desc','route_type','route_color','route_text_color','exact_times']]
    routes.to_csv(outputFolderPath + 'routes.txt', index=False)
    
    # trips.txt
    (trips, startTimes, endTimes) = getTripsTxt(schedules)
    trips.to_csv(outputFolderPath + 'trips.txt', index=False)
    
    # shapes.txt
    shapes = shapes.loc[shapes['shape_id'].isin(trips['shape_id'])]
    shapes.to_csv(outputFolderPath + 'shapes.txt', index=False)
    
    stopsByRoute = stopsByRoute.loc[stopsByRoute['shape_id'].isin(shapes['shape_id'])]
    
    # stoptimes.txt
    tripsForStopTimes = trips[['route_id','trip_id','shape_id']]
    startTimes.set_index('route', inplace=True)
    endTimes.set_index('route', inplace=True)
    tripsForStopTimes['tripNumber'] = tripsForStopTimes['trip_id'].str.extract(r'(?:_T)(\d+)').astype(int)
    tripsForStopTimes['startTime'] = startTimes.lookup(tripsForStopTimes['shape_id'], tripsForStopTimes['tripNumber'])
    tripsForStopTimes['endTime'] = endTimes.lookup(tripsForStopTimes['shape_id'], tripsForStopTimes['tripNumber'])
    tripsForStopTimes['duration'] = ((tripsForStopTimes['endTime'] - tripsForStopTimes['startTime']).dt.total_seconds()/60).round(0)
    
    shapeLength = shapes[['shape_id','shape_dist_traveled']].groupby(['shape_id'], as_index=False).max()
    shapeLength.columns = ['shape_id','length_m']
    shapeLength['shape_id'] = shapeLength['shape_id'].str.strip()
    tripsForStopTimes['shape_id'] = tripsForStopTimes['shape_id'].str.strip()
    tripsForStopTimes = tripsForStopTimes.merge(shapeLength, how='left', on='shape_id')
    tripsForStopTimes['speed_kmph'] = ((tripsForStopTimes['length_m']*60)/(tripsForStopTimes['duration']*1000))
    tripsForStopTimes = tripsForStopTimes.loc[tripsForStopTimes['length_m'].isnull()==False]
    
    stopTimes = stopsByRoute[['shape_id','stop_id','distAlongR']]
    stopTimes = stopTimes.sort_values(['shape_id','distAlongR']).reset_index(drop=True)
    stopTimes['stop_sequence'] = stopTimes.groupby('shape_id')['distAlongR'].rank(ascending=True).astype(int)
    stopTimes = stopTimes.sort_values(['shape_id','stop_sequence','stop_id']).drop_duplicates(subset=['shape_id','distAlongR'],keep='first').reset_index(drop=True)
    stopTimes['stop_sequence'] = stopTimes.groupby('shape_id')['distAlongR'].rank(ascending=True).astype(int)
    stopTimes = pd.merge(tripsForStopTimes[['shape_id','trip_id','speed_kmph','startTime']], stopTimes, how='left', on='shape_id')
    stopTimes.to_csv('test.csv', index=False)
    stopTimes['arrival_time'] = stopTimes.apply(lambda x: x['startTime'] + pd.Timedelta(minutes = np.round((x['distAlongR']*60/(x['speed_kmph']*1000)),0)), axis=1)
    # Assign timepoint = 1 for first and last stops, and 0 for the rest
    stopTimes['lastStop'] = stopTimes.groupby('trip_id')['stop_sequence'].transform(max)
    stopTimes['timepoint'] = stopTimes.apply(lambda x: 1 if x['stop_sequence']==1.0 or x['stop_sequence']==x['lastStop'] else 0, axis=1)
    stopTimes['arrival_time'] = pd.to_datetime(stopTimes['arrival_time']).dt.time
    stopTimes['stop_headsign'] = ''
    stopTimes['pickup_type'] = ''
    stopTimes['drop_off_type'] = ''
    stopTimes['shape_dist_traveled'] = np.round(stopTimes['distAlongR'],3) # Check distance values (to be in m)
    # Change hours from 00 to 24 for trips after midnight
    stopTimes['arrival_time'] = stopTimes['arrival_time'].astype(str)
    stopTimes['arrival_time'] = stopTimes['arrival_time'].apply(lambda x: '24'+x[2:] if x.startswith('00') else x)
    stopTimes['arrival_time'] = stopTimes['arrival_time'].apply(lambda x: '25'+x[2:] if x.startswith('01') else x)
    stopTimes['departure_time'] = stopTimes['arrival_time']
    stopTimes['stop_note'] = ''
    stopTimes['stop_id'] = stopTimes['stop_id'].astype(str)
    stopTimes = stopTimes[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','stop_headsign','pickup_type','drop_off_type','shape_dist_traveled','timepoint','stop_note']]
    stopTimes.to_csv(outputFolderPath + 'stop_times.txt', index=False)
    
    # stops.txt
    stops = stops.loc[stops['stop_id'].isin(stopTimes['stop_id'])]
    stops.to_csv(outputFolderPath + 'stops.txt', index=False)

In [10]:
def getTripsTxt(schedules):
    
    # Assumptions
    peakOffpeakTravelTimeRatio = 1.0
    peakOffpeakHeadwayRatio = 1.0
    
    schedules = schedules[['route','from','to','off_peak_tt','first_start_M_F','last_start_M_F','avg_freq_M_F','nTrips_M_F','Notes']]
    schedules.loc[:,['off_peak_tt']] = pd.to_timedelta(schedules['off_peak_tt'], unit='minutes')
    # Get peak travel time
    schedules['peak_tt'] = (peakOffpeakTravelTimeRatio*schedules['off_peak_tt']).dt.round('min') # rounding to minutes
    # Get parent routes (by removing _r if present)
    schedules['parent_route'] = schedules['route'].str.rstrip('_r')
    # Group by parent route and get unique first and last starts (frequency and travel times are same for both directions)
    def unique_non_null(s):
        return s.dropna().unique()
    parentRouteTimes = schedules[['parent_route','first_start_M_F','last_start_M_F','off_peak_tt','peak_tt','avg_freq_M_F','nTrips_M_F']].groupby(['parent_route']).agg({'first_start_M_F': unique_non_null, 'last_start_M_F': unique_non_null, 'off_peak_tt': unique_non_null, 'peak_tt': unique_non_null, 'avg_freq_M_F': max, 'nTrips_M_F': max}).reset_index(level=0)
    # This returns empty lists when first or last start is not indicated for either direction
    parentRouteTimes = parentRouteTimes.mask(parentRouteTimes.applymap(str).eq('[]')) # Convert all elements to string first, and then compare with '[]'. Finally use mask function to mark '[]' as na
    # Convert first_start and last_start to time format
    parentRouteTimes['first_start_M_F'] = pd.to_datetime(parentRouteTimes['first_start_M_F'], format='%H:%M')
    parentRouteTimes['last_start_M_F'] = pd.to_datetime(parentRouteTimes['last_start_M_F'], format='%H:%M')
    parentRouteTimes['avg_freq_M_F'] = parentRouteTimes['avg_freq_M_F'].astype(float)
    parentRouteTimes['nTrips_M_F'] = parentRouteTimes['nTrips_M_F'].astype(float)
    # Add one day to last starts after midnight
    parentRouteTimes['last_start_M_F'] = [(x+pd.DateOffset(1)) if x.hour == 0 else x for x in parentRouteTimes['last_start_M_F']]
    parentRouteTimes['last_start_M_F'].dt.day.unique()
    # Calculate total service hours (= end of last trip - start of first trip) for each route considering whether the last trip falls in the peak or off-peak periods
    # Assuming 6am to 9am as the morning peak and 5pm to 8pm as the evening peak
    amPeakStart = pd.to_datetime('06:00:00', format='%H:%M:%S')
    amPeakEnd = pd.to_datetime('09:00:00', format='%H:%M:%S')
    pmPeakStart = pd.to_datetime('17:00:00', format='%H:%M:%S')
    pmPeakEnd = pd.to_datetime('20:00:00', format='%H:%M:%S')
    parentRouteTimes['startBeforeAmPeak'] = [1 if x < amPeakStart else 0 for x in parentRouteTimes['first_start_M_F']]
    parentRouteTimes['startDuringAmPeak'] = [1 if (x >= amPeakStart and x <= amPeakEnd) else 0 for x in parentRouteTimes['first_start_M_F']]
    parentRouteTimes['startAfterAmPeak'] = [1 if x > amPeakEnd else 0 for x in parentRouteTimes['first_start_M_F']]
    parentRouteTimes['test_amPeak'] = parentRouteTimes['startBeforeAmPeak'] + parentRouteTimes['startDuringAmPeak'] + parentRouteTimes['startAfterAmPeak']
    parentRouteTimes['endBeforePmPeak'] = [1 if x < pmPeakStart else 0 for x in parentRouteTimes['last_start_M_F']]
    parentRouteTimes['endDuringPmPeak'] = [1 if (x >= pmPeakStart and x <= pmPeakEnd) else 0 for x in parentRouteTimes['last_start_M_F']]
    parentRouteTimes['endAfterPmPeak'] = [1 if x > pmPeakEnd else 0 for x in parentRouteTimes['last_start_M_F']]
    parentRouteTimes['test_pmPeak'] = parentRouteTimes['endBeforePmPeak'] + parentRouteTimes['endDuringPmPeak'] + parentRouteTimes['endAfterPmPeak']
    parentRouteTimes['service_hours'] = (parentRouteTimes['last_start_M_F'] - parentRouteTimes['first_start_M_F'] + (parentRouteTimes['off_peak_tt']*parentRouteTimes['endAfterPmPeak']) + (parentRouteTimes['off_peak_tt']*parentRouteTimes['endBeforePmPeak']) + (parentRouteTimes['peak_tt']*parentRouteTimes['endDuringPmPeak'])).dt.total_seconds()/3600
    # Calculate number of peak hours and off-peak hours
    parentRouteTimes['service_hours'] = parentRouteTimes['service_hours'].round(1)
    parentRouteTimes['amPeakHours'] = ((parentRouteTimes['startBeforeAmPeak']*3) + (parentRouteTimes['startDuringAmPeak']*((amPeakEnd-parentRouteTimes['first_start_M_F']).dt.total_seconds()/3600)) + 0).round(1)
    parentRouteTimes['pmPeakHours'] = ((parentRouteTimes['endAfterPmPeak']*3) + (parentRouteTimes['endDuringPmPeak']*((pmPeakEnd-parentRouteTimes['last_start_M_F']).dt.total_seconds()/3600)) + 0).round(1)
    parentRouteTimes['peakHours'] = (parentRouteTimes['amPeakHours'] + parentRouteTimes['pmPeakHours']).round(1)
    parentRouteTimes['offPeakHours'] = (parentRouteTimes['service_hours'] - parentRouteTimes['peakHours']).round(1)
    # Get number of trips in peak hours and off-peak hours
    # Given avg_freq is actualy offpeak headway
    # Assuming peak headway = peakOffpeakHeadwayRatio * offpeak headway
    parentRouteTimes['offPeakHeadway'] = parentRouteTimes['avg_freq_M_F']
    parentRouteTimes['peakHeadway'] = (peakOffpeakHeadwayRatio * parentRouteTimes['avg_freq_M_F'])
    parentRouteTimes['offPeakTrips'] = (parentRouteTimes['offPeakHours']*60/parentRouteTimes['offPeakHeadway']).round(0)
    parentRouteTimes['peakTrips'] = (parentRouteTimes['peakHours']*60/parentRouteTimes['peakHeadway']).round(0)
    parentRouteTimes['totalTrips'] = parentRouteTimes['peakTrips'] + parentRouteTimes['offPeakTrips']
    parentRouteTimes = parentRouteTimes[['parent_route','first_start_M_F','last_start_M_F','avg_freq_M_F','service_hours','peakHours','offPeakHours','peakTrips','offPeakTrips','totalTrips','peakHeadway','offPeakHeadway','startBeforeAmPeak','startDuringAmPeak','startAfterAmPeak','endBeforePmPeak','endDuringPmPeak','endAfterPmPeak']]
    parentRouteTimes.columns = ['parent_route','first_start_parent','last_start_parent','avg_freq_M_F','serviceHours','peakHours','offPeakHours','peakTrips','offPeakTrips','totalTrips','peakHeadway','offPeakHeadway','startBeforeAmPeak','startDuringAmPeak','startAfterAmPeak','endBeforePmPeak','endDuringPmPeak','endAfterPmPeak']
    schedules = schedules.drop(['avg_freq_M_F'], axis=1)
    schedules = schedules.merge(parentRouteTimes, how='left', on='parent_route')
    # Impute missing first and last starts from those of the parent route:
    # 1. Convert to datetime format
    schedules['first_start_M_F'] = pd.to_datetime(schedules['first_start_M_F'], format='%H:%M')
    schedules['last_start_M_F'] = pd.to_datetime(schedules['last_start_M_F'], format='%H:%M')
    # 2. Fill nas by adding or subtracting travel times (peak or off-peak) to 
    schedules['first_start_M_F'] = schedules['first_start_M_F'].fillna(schedules['first_start_parent'] + (schedules['off_peak_tt']*(schedules['startBeforeAmPeak']+schedules['startAfterAmPeak'])) + (schedules['peak_tt']*schedules['startDuringAmPeak']))
    schedules['last_start_M_F'] = schedules['last_start_M_F'].fillna(schedules['last_start_parent'] - (schedules['off_peak_tt']*(schedules['endBeforePmPeak']+schedules['endAfterPmPeak'])) - (schedules['peak_tt']*schedules['endDuringPmPeak']))
    # Peak headway is null when there are no peak trips - replace with 0 and convert to int
    schedules['peakHeadway'] = schedules['peakHeadway'].fillna(0)
    schedules['peakHeadway'] = schedules['peakHeadway'].astype(float)
    schedules['offPeakHeadway'] = schedules['offPeakHeadway'].astype(float)
    schedules['totalTrips'] = schedules['totalTrips'].astype(int)
    schedules = schedules.drop(['startBeforeAmPeak','startDuringAmPeak','startAfterAmPeak','endBeforePmPeak','endDuringPmPeak','endAfterPmPeak'], axis=1)
    # Get trip start and end times
    schedules['startTimes'] = schedules.apply(generateListOfStartTimes, axis=1)
    schedules['endTimes'] = schedules.apply(generateListOfEndTimes, axis=1)
    schedules['nStartTimes'] = [len(x) for x in schedules['startTimes']]
    schedules['nEndTimes'] = [len(x) for x in schedules['endTimes']]
    schedules['diffInTrips'] = schedules['nStartTimes'] - schedules['totalTrips']
    schedules['lastStartTime'] = [x[-1] for x in schedules['startTimes']]
    schedules['diffInLastStart'] = pd.to_datetime(schedules['lastStartTime'], format='%H:%M:%S') - schedules['last_start_M_F']
    startTimes = schedules['startTimes'].apply(pd.Series)
    startTimes = startTimes.rename(columns = lambda x: int(x+1))
    startTimes = pd.concat([schedules[['route']], startTimes], axis=1)
    endTimes = schedules['endTimes'].apply(pd.Series)
    endTimes = endTimes.rename(columns = lambda x: int(x+1))
    endTimes = pd.concat([schedules[['route']], endTimes], axis=1)
    # Generate trips.txt
    trips = schedules[['route','from','to','parent_route','nStartTimes']]
    trips['tripNumbers'] = [list(range(1,n+1)) for n in trips['nStartTimes']]
    trips = pd.DataFrame({col:np.repeat(trips[col].values, trips['tripNumbers'].str.len()) for col in trips.columns.drop('tripNumbers')}).assign(tripNumbers=np.concatenate(trips['tripNumbers'].values))
    trips['route_id'] = trips['route']
    trips['trip_id'] = trips['route'] + '_T' + trips['tripNumbers'].map(str)
    trips['service_id'] = 'mtwtfss'
    trips['trip_headsign'] = trips['to']
    trips['direction_id'] = np.where(trips['route'].str.endswith('_r'),1,0)
    trips['block_id'] = ''
    trips['wheelchair_accessible'] = ''
    trips['route_direction'] = trips['from'] + ' - ' + trips['to']
    trips['trip_note'] = ''
    trips['bikes_allowed'] = ''
    trips.rename(columns={'route':'shape_id'}, inplace = True)
    trips = trips.sort_values(['route_id','shape_id','tripNumbers']).reset_index(drop=True)
    trips = trips[['route_id','service_id','trip_id','shape_id','trip_headsign','direction_id','block_id','wheelchair_accessible','route_direction','trip_note','bikes_allowed']]
    return (trips,startTimes,endTimes)

In [11]:
# Function to generate a list of start times

def generateListOfStartTimes(row):
    
    amPeakStart = pd.to_datetime('06:00:00', format='%H:%M:%S')
    amPeakEnd = pd.to_datetime('09:00:00', format='%H:%M:%S')
    pmPeakStart = pd.to_datetime('17:00:00', format='%H:%M:%S')
    pmPeakEnd = pd.to_datetime('20:00:00', format='%H:%M:%S')
    
    t = row.first_start_M_F
    limit = row.last_start_M_F
    if limit.hour == 0 and limit.day == 1:
        limit = limit + pd.Timedelta(days=1) # Add a day if after midnight
    startTimes = []
        
    for i in range(0,row.totalTrips):
        if t <= limit:
            startTimes.append(datetime.datetime.strptime(str(t), "%Y-%m-%d %H:%M:%S"))
        if t < amPeakStart:
            next_t = t + datetime.timedelta(seconds=row.offPeakHeadway*60)
            t = next_t
        elif t >= amPeakStart and t < amPeakEnd:
            next_t = t + datetime.timedelta(seconds=row.peakHeadway*60)
            t = next_t
        elif t >= amPeakEnd and t< pmPeakStart:
            next_t = t + datetime.timedelta(seconds=row.offPeakHeadway*60)
            t = next_t
        elif t >= pmPeakStart and t < pmPeakEnd:
            next_t = t + datetime.timedelta(seconds=row.peakHeadway*60)
            t = next_t
        elif t >= pmPeakEnd:
            next_t = t + datetime.timedelta(seconds=row.offPeakHeadway*60)
            t = next_t
        else:
            break
        
    return startTimes

# Function to generate a list of trip end times based on the start times

def generateListOfEndTimes(row):
    
    amPeakStart = pd.to_datetime('06:00:00', format='%H:%M:%S')
    amPeakEnd = pd.to_datetime('09:00:00', format='%H:%M:%S')
    pmPeakStart = pd.to_datetime('17:00:00', format='%H:%M:%S')
    pmPeakEnd = pd.to_datetime('20:00:00', format='%H:%M:%S')
    
    startTimes = row.startTimes
    endTimes = []
    
    for t in startTimes:
        if t < amPeakStart:
            endTimes.append(t + row.off_peak_tt)
        elif t >= amPeakStart and t < amPeakEnd:
            endTimes.append(t + row.peak_tt)
        elif t >= amPeakEnd and t < pmPeakStart:
            endTimes.append(t + row.off_peak_tt)
        elif t >= pmPeakStart and t < pmPeakEnd:
            endTimes.append(t + row.peak_tt)
        elif t >= pmPeakEnd:
            endTimes.append(t + row.off_peak_tt)
    
    return endTimes


In [12]:
getRoutesTripsStoptimes(schedules, shapes, stopsByRoute, stops)

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import sys
/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/pandas/core/indexing.py:966: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[item] = s
/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =